Universidad Torcuato Di Tella

Licenciatura en Tecnología Digital\
**Tecnología Digital VI: Inteligencia Artificial**

# **XGBoost**

En esta *notebook*, crearemos un modelo usando XGBoost, para tratar de predecir decisiones bancarias de clientes.

XGBoost viene de eXtreme Gradient Boosting y es una librería de árboles de decisión impulsados por gradiente.

XGBoost se importa usando el paquete `xgboost` y se suele usar el alias `xgb`.

In [1]:
# !pip install -U xgboost

import xgboost as xgb

ModuleNotFoundError: No module named 'xgboost'

También importamos otras utilidades necesarias:

In [ ]:
import pandas as pd # Para cargar los datos y hacer OHE.
import numpy as np  # Para lidiar con NaNs.
import time
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, make_scorer
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import confusion_matrix

random_state = 42
np.random.seed(random_state)

### **Carga del *data frame***

Cargamos un archivo CSV que tiene datos bancarios y una variable predictora yes/no que es la columna `y` e indica si el cliente se suscribió o no a un depósito a plazo. Fuente: https://archive.ics.uci.edu/dataset/222/bank+marketing. Ya está disponible en el Campus Virtual, en la sección `Datasets`.

In [ ]:
df = pd.read_csv('./bank-full.csv', sep = ';')

In [ ]:
len(df)

45211

In [ ]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


Miramos las columnas numéricas:

In [ ]:
df.describe()

,age,balance,day,duration,campaign,pdays,previous
count,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000
mean,40.936210,1362.272058,15.806419,258.163080,2.763841,40.197828,0.580323
std,10.618762,3044.765829,8.322476,257.527812,3.098021,100.128746,2.303441
min,18.000000,-8019.000000,1.000000,0.000000,1.000000,-1.000000,0.000000
25%,33.000000,72.000000,8.000000,103.000000,1.000000,-1.000000,0.000000
50%,39.000000,448.000000,16.000000,180.000000,2.000000,-1.000000,0.000000
75%,48.000000,1428.000000,21.000000,319.000000,3.000000,-1.000000,0.000000
max,95.000000,102127.000000,31.000000,4918.000000,63.000000,871.000000,275.000000


Miramos las columnas tipo `object` (en este caso, son todas categóricas, encodeadas como *strings*):

In [ ]:
df.describe(include = 'object')

,job,marital,education,default,housing,loan,contact,month,poutcome,y
count,45211,45211,45211,45211,45211,45211,45211,45211,45211,45211
unique,12,3,4,2,2,2,3,12,4,2
top,blue-collar,married,secondary,no,yes,no,cellular,may,unknown,no
freq,9732,27214,23202,44396,25130,37967,29285,13766,36959,39922


Investigamos los tipos de cada columna:

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB


### **Valores faltantes**

XGBoost puede trabajar con valores faltantes. Agregamos algunos en el data frame para demostrarlo.

In [ ]:
probability = 0.2
mask = np.random.rand(*df.shape) < probability
# Removemos el ruido de la columna 'y', ya que no queremos agregar datos faltantes en la variable predictora.
mask[:, mask.shape[1] - 1] = False
df[mask] = np.nan

Podemos observar los datos faltantes como valores NaN:

In [ ]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58.0,management,married,tertiary,NaN,NaN,NaN,no,unknown,5.0,NaN,261.0,1.0,-1.0,NaN,NaN,no
1,44.0,technician,single,secondary,NaN,29.0,yes,no,unknown,NaN,may,151.0,NaN,-1.0,NaN,NaN,no
2,33.0,entrepreneur,married,NaN,no,2.0,NaN,yes,NaN,5.0,may,76.0,1.0,-1.0,0.0,NaN,no
3,47.0,blue-collar,married,unknown,no,NaN,NaN,NaN,unknown,5.0,may,92.0,1.0,-1.0,0.0,NaN,no
4,NaN,unknown,single,NaN,NaN,1.0,no,no,unknown,NaN,may,NaN,1.0,-1.0,0.0,NaN,no


XGBoost no solía soportar variables categóricas, pero actualmente las soporta de forma experimental (fuente: https://xgboost.readthedocs.io/en/stable/tutorials/categorical.html).
        
Sin embargo, en este caso, usaremos one-hot encoding.

In [ ]:
# Importante: ¡sólo poner las categóricas y excluir la variable a predecir!
pd_ohe = pd.get_dummies(df,
                        columns = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome'],
                        sparse = True,    # Devolver una matriz rala.
                        dummy_na = False, # No agregar columna para NaNs.
                        dtype = int       # XGBoost no trabaja con 'object'; necesitamos que sean numéricos.
                       )
pd_ohe

,age,balance,day,duration,campaign,pdays,previous,y,job_admin.,job_blue-collar,...,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_failure,poutcome_other,poutcome_success,poutcome_unknown
0,58.0,NaN,5.0,261.0,1.0,-1.0,NaN,no,0,0,...,0,0,0,0,0,0,0,0,0,0
1,44.0,29.0,NaN,151.0,NaN,-1.0,NaN,no,0,0,...,0,0,1,0,0,0,0,0,0,0
2,33.0,2.0,5.0,76.0,1.0,-1.0,0.0,no,0,0,...,0,0,1,0,0,0,0,0,0,0
3,47.0,NaN,5.0,92.0,1.0,-1.0,0.0,no,0,1,...,0,0,1,0,0,0,0,0,0,0
4,NaN,1.0,NaN,NaN,1.0,-1.0,0.0,no,0,0,...,0,0,1,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,51.0,825.0,17.0,977.0,NaN,NaN,NaN,yes,0,0,...,0,0,0,1,0,0,0,0,0,0
45207,71.0,1729.0,17.0,456.0,NaN,-1.0,0.0,yes,0,0,...,0,0,0,1,0,0,0,0,0,1
45208,72.0,5715.0,17.0,1127.0,5.0,184.0,3.0,yes,0,0,...,0,0,0,1,0,0,0,0,1,0
45209,57.0,668.0,17.0,508.0,4.0,-1.0,0.0,no,0,1,...,0,0,0,1,0,0,0,0,0,1


In [ ]:
len(pd_ohe.columns)

52

In [ ]:
pd_ohe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 52 columns):
 #   Column               Non-Null Count  Dtype           
---  ------               --------------  -----           
 0   age                  36174 non-null  float64         
 1   balance              36059 non-null  float64         
 2   day                  36255 non-null  float64         
 3   duration             36225 non-null  float64         
 4   campaign             36270 non-null  float64         
 5   pdays                36255 non-null  float64         
 6   previous             36170 non-null  float64         
 7   y                    45211 non-null  object          
 8   job_admin.           45211 non-null  Sparse[int64, 0]
 9   job_blue-collar      45211 non-null  Sparse[int64, 0]
 10  job_entrepreneur     45211 non-null  Sparse[int64, 0]
 11  job_housemaid        45211 non-null  Sparse[int64, 0]
 12  job_management       45211 non-null  Sparse[int64, 0]
 13  j

Comparación con matriz no rala:

In [ ]:
pd.get_dummies(df,
               columns = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome'],
               sparse = False,   # Devolver una matriz densa.
               dummy_na = False, # No agregar columna para NaNs.
               dtype = int       # XGBoost no trabaja con 'object'; necesitamos que sean numéricos.
              ).info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 52 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   age                  36174 non-null  float64
 1   balance              36059 non-null  float64
 2   day                  36255 non-null  float64
 3   duration             36225 non-null  float64
 4   campaign             36270 non-null  float64
 5   pdays                36255 non-null  float64
 6   previous             36170 non-null  float64
 7   y                    45211 non-null  object 
 8   job_admin.           45211 non-null  int64  
 9   job_blue-collar      45211 non-null  int64  
 10  job_entrepreneur     45211 non-null  int64  
 11  job_housemaid        45211 non-null  int64  
 12  job_management       45211 non-null  int64  
 13  job_retired          45211 non-null  int64  
 14  job_self-employed    45211 non-null  int64  
 15  job_services         45211 non-null 

6 MB vs. 18 MB, aproximadamente.

### **Preparar conjuntos de entrenamiento, validación (*hold-out*) y evaluación**

Para este ejemplo, usaremos conjuntos de datos de *train*, *validation* y *test* fijos. Es decir, para validación usaremos un hold-out *set*. Decidimos esto porque usar *cross-validation* es más costoso.

In [ ]:
y = pd_ohe[['y']].copy() # Usamos copy para no modificar un view abajo, ya que genera un warning.
y

,y
0,no
1,no
2,no
3,no
4,no
...,...
45206,yes
45207,yes
45208,yes
45209,no


In [ ]:
y[y['y'] == 'yes'] = 1
y[y['y'] == 'no'] = 0
y['y'] = y['y'].astype(int)

In [ ]:
y['y'].unique()

array([0, 1])

In [ ]:
X = pd_ohe.drop('y', axis = 1)
X

,age,balance,day,duration,campaign,pdays,previous,job_admin.,job_blue-collar,job_entrepreneur,...,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_failure,poutcome_other,poutcome_success,poutcome_unknown
0,58.0,NaN,5.0,261.0,1.0,-1.0,NaN,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,44.0,29.0,NaN,151.0,NaN,-1.0,NaN,0,0,0,...,0,0,1,0,0,0,0,0,0,0
2,33.0,2.0,5.0,76.0,1.0,-1.0,0.0,0,0,1,...,0,0,1,0,0,0,0,0,0,0
3,47.0,NaN,5.0,92.0,1.0,-1.0,0.0,0,1,0,...,0,0,1,0,0,0,0,0,0,0
4,NaN,1.0,NaN,NaN,1.0,-1.0,0.0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,51.0,825.0,17.0,977.0,NaN,NaN,NaN,0,0,0,...,0,0,0,1,0,0,0,0,0,0
45207,71.0,1729.0,17.0,456.0,NaN,-1.0,0.0,0,0,0,...,0,0,0,1,0,0,0,0,0,1
45208,72.0,5715.0,17.0,1127.0,5.0,184.0,3.0,0,0,0,...,0,0,0,1,0,0,0,0,1,0
45209,57.0,668.0,17.0,508.0,4.0,-1.0,0.0,0,1,0,...,0,0,0,1,0,0,0,0,0,1


In [ ]:
val_test_size = 0.3 # Proporción de la suma del test de validación y del de test.
X_train, X_tmp, Y_train, Y_tmp = train_test_split(X, y,
                                                  train_size = 0.7,
                                                  random_state = random_state,
                                                  stratify = y)

In [ ]:
X_val, X_test, Y_val, Y_test = train_test_split(X_tmp, Y_tmp,
                                                train_size = 0.5,
                                                random_state = random_state,
                                                stratify = Y_tmp)

In [ ]:
print(f'Cantidad de datos de train: {len(X_train)}')
print(f'Cantidad de datos de validación: {len(X_val)}')
print(f'Cantidad de datos de test: {len(X_test)}')

Cantidad de datos de train: 31647
Cantidad de datos de validación: 6782
Cantidad de datos de test: 6782


### **Ahora sí, a usar XGBoost**

Para usar XGBoost, creamos una instancia de la clase `XGBClassifier` y le especificamos el tipo como categórico (el parámetro `objective`). Además, podemos especificar otros parámetros típicos de XGBoost. Enlace a la documentación: https://xgboost.readthedocs.io/en/stable/parameter.html#parameters-for-tree-booster.

In [ ]:
clf_xgb = xgb.XGBClassifier(objective = 'binary:logistic',
                            seed = random_state,
                            eval_metric = 'auc')

Para entrenar el modelo, podemos llamar al método `fit` como con los otros tipos de modelo.

In [ ]:
clf_xgb.fit(X_train, Y_train, verbose = True, eval_set = [(X_val, Y_val)])

[0]	validation_0-auc:0.82470
[1]	validation_0-auc:0.84480
[2]	validation_0-auc:0.86397
[3]	validation_0-auc:0.86478
[4]	validation_0-auc:0.87343
[5]	validation_0-auc:0.88123
[6]	validation_0-auc:0.88373
[7]	validation_0-auc:0.88562
[8]	validation_0-auc:0.88861
[9]	validation_0-auc:0.88933
[10]	validation_0-auc:0.89103
[11]	validation_0-auc:0.89199
[12]	validation_0-auc:0.89284
[13]	validation_0-auc:0.89238
[14]	validation_0-auc:0.89273
[15]	validation_0-auc:0.89371
[16]	validation_0-auc:0.89430
[17]	validation_0-auc:0.89491
[18]	validation_0-auc:0.89708
[19]	validation_0-auc:0.89729
[20]	validation_0-auc:0.89744
[21]	validation_0-auc:0.89793
[22]	validation_0-auc:0.89872
[23]	validation_0-auc:0.89881
[24]	validation_0-auc:0.89882
[25]	validation_0-auc:0.89914
[26]	validation_0-auc:0.89883
[27]	validation_0-auc:0.89891
[28]	validation_0-auc:0.89902
[29]	validation_0-auc:0.89944
[30]	validation_0-auc:0.89968
[31]	validation_0-auc:0.89960
[32]	validation_0-auc:0.89950
[33]	validation_0-au

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              gamma=None, gpu_id=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              n_estimators=100, n_jobs=None, num_parallel_tree=None,
              predictor=None, random_state=None, ...)

### **Búsqueda de hiperparámetros**

No queremos buscar hiperparámetros con cross-validation, porque tardaría mucho. Hacemos *random search* manualmente con la clase `ParameterSampler`. Aclaración: también existe la clase `ParameterGrid` por si quisiéramos hacer lo mismo con *grid search*. Definimos los posibles valores:

In [ ]:
from scipy.stats import uniform
params = {'max_depth': list(range(1, 40)),
          'learning_rate': uniform(scale = 0.2),
          'gamma': uniform(scale = 2),
          'reg_lambda': uniform(scale = 5),        # Parámetro de regularización.
          'subsample': uniform(0.5, 0.5),          # Entre 0.5 y 1.
          'min_child_weight': uniform(scale = 5),
          'colsample_bytree': uniform(0.75, 0.25), # Entre 0.75 y 1.
          'n_estimators': list(range(1, 1000))
         }

Recordemos que podemos ver las definiciones de los hiperparámetros acá: https://xgboost.readthedocs.io/en/stable/parameter.html.

In [ ]:
start = time.time()
best_score = 0
best_estimator = None
iterations = 100
for g in ParameterSampler(params, n_iter = iterations, random_state = random_state):
    clf_xgb = xgb.XGBClassifier(objective = 'binary:logistic', seed = random_state, eval_metric = 'auc', **g)
    clf_xgb.fit(X_train, Y_train, eval_set = [(X_val, Y_val)], verbose = False)

    y_pred = clf_xgb.predict_proba(X_val)[:, 1] # Obtenemos la probabilidad de una de las clases (cualquiera).
    auc_roc = sklearn.metrics.roc_auc_score(Y_val, y_pred)
    # Guardamos si es mejor.
    if auc_roc > best_score:
        print(f'Mejor valor de ROC-AUC encontrado: {auc_roc}')
        best_score = auc_roc
        best_grid = g
        best_estimator = clf_xgb

end = time.time()
print('ROC-AUC: %0.5f' % best_score)
print('Grilla:', best_grid)
print(f'Tiempo transcurrido: {str(end - start)} segundos')
print(f'Tiempo de entrenamiento por iteración: {str(round((end - start) / iterations, 2))} segundos')

Mejor valor de ROC-AUC encontrado: 0.8871468798217762
Mejor valor de ROC-AUC encontrado: 0.8966518259020139
Mejor valor de ROC-AUC encontrado: 0.901884162952269
Mejor valor de ROC-AUC encontrado: 0.9035085283917962
ROC-AUC: 0.90351
Grilla: {'colsample_bytree': 0.7626921327598493, 'gamma': 1.7732342979013198, 'learning_rate': 0.005523354374740941, 'max_depth': 34, 'min_child_weight': 0.469909699204345, 'n_estimators': 727, 'reg_lambda': 3.360130676475997, 'subsample': 0.664076333737366}
Tiempo transcurrido: 2191.9723739624023 segundos
Tiempo de entrenamiento por iteración: 21.92 segundos


¿Cuánto tardaríamos en hacer un grid search con la misma escala?

Si sólo tuviéramos 5 opciones fijas en cada parámetro, teniendo 7 parámetros, serían 546.875 multiplicado por el tiempo que dure cada iteración. Si cada iteración dura 1 segundo... esto es (sin usar paralelismo) ¡6 días!

In [ ]:
best_grid = {'colsample_bytree': 0.7626921327598493,
             'gamma': 1.7732342979013198,
             'learning_rate': 0.005523354374740941,
             'max_depth': 34,
             'min_child_weight': 0.469909699204345,
             'n_estimators': 727,
             'reg_lambda': 3.360130676475997,
             'subsample': 0.664076333737366
            }
# Aclaración: acá está "hardcodeado", pero se puede hacer mejor, accediendo a los valores de `best_grid`.

best_estimator = xgb.XGBClassifier(objective = 'binary:logistic',
                                   seed = random_state,
                                   eval_metric = 'auc',
                                   **best_grid)

best_estimator.fit(X_train, Y_train, verbose = True,  eval_set = [(X_val, Y_val)])

# roc_auc_score requiere un array 1D; da lo mismo qué dimensión le pasemos: 90 o 1.
y_pred = best_estimator.predict_proba(X_val)[:, 1]
auc_roc = sklearn.metrics.roc_auc_score(Y_val, y_pred)
print('AUC-ROC validación: %0.5f' % auc_roc)

[0]	validation_0-auc:0.84266
[1]	validation_0-auc:0.85949
[2]	validation_0-auc:0.86934
[3]	validation_0-auc:0.87212
[4]	validation_0-auc:0.87541
[5]	validation_0-auc:0.87659
[6]	validation_0-auc:0.87717
[7]	validation_0-auc:0.88326
[8]	validation_0-auc:0.88335
[9]	validation_0-auc:0.88287
[10]	validation_0-auc:0.88236
[11]	validation_0-auc:0.88298
[12]	validation_0-auc:0.88353
[13]	validation_0-auc:0.88361
[14]	validation_0-auc:0.88399
[15]	validation_0-auc:0.88351
[16]	validation_0-auc:0.88380
[17]	validation_0-auc:0.88381
[18]	validation_0-auc:0.88371
[19]	validation_0-auc:0.88446
[20]	validation_0-auc:0.88594
[21]	validation_0-auc:0.88642
[22]	validation_0-auc:0.88842
[23]	validation_0-auc:0.88892
[24]	validation_0-auc:0.88930
[25]	validation_0-auc:0.88911
[26]	validation_0-auc:0.88921
[27]	validation_0-auc:0.88913
[28]	validation_0-auc:0.88956
[29]	validation_0-auc:0.88969
[30]	validation_0-auc:0.88983
[31]	validation_0-auc:0.88984
[32]	validation_0-auc:0.88991
[33]	validation_0-au

Una vez entrenado, podemos observar atributos de cada campo como, por ejemplo, el `cover`:

In [ ]:
bst = best_estimator.get_booster()
for importance_type in ('weight', 'gain', 'cover', 'total_gain', 'total_cover'):
    print('%s: ' % importance_type, bst.get_score(importance_type = importance_type))
    print('--------------')

weight:  {'age': 20313.0, 'balance': 25582.0, 'day': 20270.0, 'duration': 23876.0, 'campaign': 9372.0, 'pdays': 10966.0, 'previous': 6781.0, 'job_admin.': 1395.0, 'job_blue-collar': 1331.0, 'job_entrepreneur': 554.0, 'job_housemaid': 468.0, 'job_management': 1816.0, 'job_retired': 895.0, 'job_self-employed': 649.0, 'job_services': 746.0, 'job_student': 1009.0, 'job_technician': 1667.0, 'job_unemployed': 654.0, 'job_unknown': 62.0, 'marital_divorced': 1348.0, 'marital_married': 1730.0, 'marital_single': 1905.0, 'education_primary': 945.0, 'education_secondary': 1709.0, 'education_tertiary': 2038.0, 'education_unknown': 922.0, 'default_no': 1620.0, 'default_yes': 279.0, 'housing_no': 1988.0, 'housing_yes': 1565.0, 'loan_no': 1852.0, 'loan_yes': 1068.0, 'contact_cellular': 2082.0, 'contact_telephone': 977.0, 'contact_unknown': 1233.0, 'month_apr': 1944.0, 'month_aug': 1935.0, 'month_dec': 900.0, 'month_feb': 2406.0, 'month_jan': 926.0, 'month_jul': 1418.0, 'month_jun': 1824.0, 'month_mar'

### **Conjunto de test**

Para finalizar, ejecutamos el modelo para el conjunto de test, para tener una mejor estimación de cómo se comportaría el modelo en un escenario de producción, con un conjunto de datos que el modelo nunca ha visto y que tampoco se ha usado para tomar decisiones.

In [ ]:
y_pred = best_estimator.predict_proba(X_test)[:, 1]
auc_roc = sklearn.metrics.roc_auc_score(Y_test, y_pred)
print('AUC-ROC test: %0.5f' % auc_roc)

AUC-ROC test: 0.89681


Efectivamente, vemos que la *performance* es ligeramente menor en el conjunto de test.